# Network Traffic Classification Using Transformer-Enhanced LSTM Models

This notebook runs the proposed Transformer-Enhanced LSTM model only. Published baseline results are used for discussion in the research paper, but no baseline model is trained inside this project.

## 1. Setup

Import the reusable project pipeline from the `src/` directory.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"

sys.path.insert(0, str(SRC_DIR))

from preprocessing import prepare_dataset
from train import run_project

print(PROJECT_ROOT)

## 2. Experiment Settings

Choose `cic`, `unsw`, or `all`. The model uses EarlyStopping and ReduceLROnPlateau, so a larger epoch value is safe; the best validation-loss weights are restored.

In [ ]:
RUN_MODE = 'all'  # options: 'cic', 'unsw', 'all'
EPOCHS = 50
BATCH_SIZE = 128
MAX_SAMPLES = None  # use a small number only for quick testing
CLASS_WEIGHT_MODE = 'auto'  # options: 'auto', 'sqrt', 'balanced', 'none'
SAVE_MODEL = False
SEED = 42

## 3. Preview Preprocessing

This preview confirms feature shape, class labels, and split distributions. It uses a small sample and does not affect final training.

In [ ]:
preview_datasets = ['cic', 'unsw'] if RUN_MODE == 'all' else [RUN_MODE]

for dataset_key in preview_datasets:
    data = prepare_dataset(dataset=dataset_key, data_dir=PROJECT_ROOT / 'dataset', max_samples=1000)
    print(f'\nDataset: {data.dataset_name}')
    print('Label column:', data.label_column)
    print('Input shape:', data.input_shape)
    print('Classes:', data.class_names)
    print('Train tensor:', data.X_train.shape)
    print('Validation tensor:', data.X_val.shape)
    print('Test tensor:', data.X_test.shape)
    print('Class distribution:', data.class_distribution)

## 4. Train and Evaluate Proposed Model

This cell trains only the Transformer-Enhanced LSTM and writes proposed-model metrics and figures.

In [ ]:
if RUN_MODE not in {'cic', 'unsw', 'all'}:
    raise ValueError("RUN_MODE must be one of: 'cic', 'unsw', 'all'")

experiments = run_project(
    dataset='cic' if RUN_MODE == 'all' else RUN_MODE,
    all_datasets=(RUN_MODE == 'all'),
    data_dir=PROJECT_ROOT / 'dataset',
    results_dir=PROJECT_ROOT / 'results',
    figures_dir=PROJECT_ROOT / 'figures',
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    max_samples=MAX_SAMPLES,
    class_weight_mode=CLASS_WEIGHT_MODE,
    save_model=SAVE_MODEL,
    seed=SEED,
)

list(experiments.keys())

## 5. Display Metrics

In [ ]:
metrics_file = {
    'cic': PROJECT_ROOT / 'results' / 'cic_darknet2020_metrics.txt',
    'unsw': PROJECT_ROOT / 'results' / 'unsw_nb15_metrics.txt',
    'all': PROJECT_ROOT / 'results' / 'metrics.txt',
}[RUN_MODE]

print(metrics_file)
print(metrics_file.read_text(encoding='utf-8'))

## 6. Display Figures

In [ ]:
from IPython.display import Image, display

prefix = {'cic': 'cic_darknet2020_', 'unsw': 'unsw_nb15_', 'all': ''}[RUN_MODE]
figure_paths = [
    PROJECT_ROOT / 'results' / f'{prefix}confusion_matrix.png',
    PROJECT_ROOT / 'figures' / f'{prefix}accuracy_curve.png',
    PROJECT_ROOT / 'figures' / f'{prefix}loss_curve.png',
]
if RUN_MODE == 'all':
    figure_paths.append(PROJECT_ROOT / 'figures' / 'dataset_comparison_graph.png')


for path in figure_paths:
    print(path)
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print('Missing file. Run the training cell first.')

## 7. Summary Table

In [ ]:
rows = []
for dataset_key, experiment in experiments.items():
    result = experiment['metrics']
    rows.append({
        'Dataset': experiment['dataset_name'],
        'Model': result['model_name'],
        'Accuracy': result['accuracy'],
        'Weighted F1': result['weighted_f1'],
        'Macro F1': result['macro_f1'],
        'Weighted AUC': result['weighted_auc'],
    })

try:
    import pandas as pd
    display(pd.DataFrame(rows))
except Exception:
    rows